# Multi-LLM-as-Judge: ADRD Medical AI
## Complete SMU SuperPOD Experiment Notebook

**Tailored to:** `mkotha` on SMU SuperPOD | Account: `xnluo_ai_chatbot_cognitive_0002`

| Phase | Cell | What happens |
|---|---|---|
| 0 | — | SSH + SOCKS proxy + tmux + srun (local terminal) |
| 1 | 1 | Verify GPU node, venv, paths |
| 2 | 2 | Install deps + tabulate fix |
| 3 | 3 | Load .env + HF cache |
| 3 | 4 | Download all 4 judge models |
| 4 | 5 | Kill any old vLLM + launch 4 fresh servers |
| 4 | 6 | Tail logs (optional check) |
| 5 | 7 | Health check all 4 judges (20 min timeout) |
| 6 | 8 | Repo structure check |
| 6 | 9 | Unit tests |
| 7 | 10 | Exp 1 — Dataset analysis |
| 8 | 11 | Exp 2 — Agreement analysis |
| 9 | 12 | Exp 3 — Rubric sensitivity |
| 10 | 13 | Exp 4 — Box plots |
| 11 | 14 | Results summary + CSV |
| 12 | 15 | Display figures inline |
| 13 | 16 | Push to GitHub |
| 14 | 17 | Cleanup / kill vLLM |

---

## Phase 0 — Terminal Commands (run BEFORE opening this notebook)

```bash
# 1. On your LOCAL machine:
ssh -C -D 8080 mkotha@superpod.smu.edu

# 2. On SuperPOD login node:
tmux new -s cs7325
# (to reattach later: tmux attach -t cs7325)

# 3. Request GPU node (inside tmux):
srun -A xnluo_ai_chatbot_cognitive_0002 -N1 -G2 -c32 --mem=192G --time=4:00:00 --pty $SHELL

# 4. On the GPU compute node:
cd /lustre/smuexa01/client/users/mkotha/CS7325

# 5. Clone repo (only first time):
git clone https://YOUR_GITHUB_TOKEN@github.com/m22oct2000/Multi_LLM_as_Judge_Medical_AI.git
cd Multi_LLM_as_Judge_Medical_AI

# 6. Create .env file (only first time):
cat > .env << 'EOF'
HF_TOKEN=hf_xxxxxxxxxxxx
GITHUB_TOKEN=ghp_xxxxxxxxxxxx
LOCAL_VLLM_JUDGE1_URL=http://localhost:8001
LOCAL_VLLM_JUDGE2_URL=http://localhost:8002
LOCAL_VLLM_JUDGE3_URL=http://localhost:8003
LOCAL_VLLM_JUDGE4_URL=http://localhost:8004
EOF

# 7. Activate venv and launch JupyterLab:
source /lustre/smuexa01/client/users/mkotha/CS7325/.venv/bin/activate
cd /lustre/smuexa01/client/users/mkotha/CS7325/Multi_LLM_as_Judge_Medical_AI
jupyter lab --ip=0.0.0.0 --no-browser
# Open the printed URL in your browser (SOCKS proxy port 8080)
```


---
## Phase 1 — Verify GPU Node & Environment

In [1]:
# CELL 1: Verify GPU, venv, paths
import subprocess, os, sys, importlib
from pathlib import Path
from datetime import datetime

def shell(cmd, check=False):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (result.stdout + result.stderr).strip()
    if out: print(out)
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed: {cmd}')
    return result.returncode

LUSTRE_BASE = Path('/lustre/smuexa01/client/users/mkotha/CS7325')
ROOT        = LUSTRE_BASE / 'Multi_LLM_as_Judge_Medical_AI'
VENV        = LUSTRE_BASE / '.venv'
HF_CACHE    = LUSTRE_BASE / 'hf_models'
LOG_DIR     = LUSTRE_BASE / 'vllm_logs'
PYTHON      = str(VENV / 'bin' / 'python')

HF_CACHE.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f'Repo root : {ROOT}')
print(f'venv      : {VENV}')
print(f'HF cache  : {HF_CACHE}')
print(f'Python    : {sys.executable}')
print(f'Started   : {datetime.now().isoformat()}')

print('\n=== GPU Status (expect 2x A100 80GB) ===')
shell('nvidia-smi --query-gpu=index,name,memory.total,memory.free --format=csv,noheader')

print('\n=== SLURM allocation ===')
shell('echo "Node: $SLURMD_NODENAME  Job: $SLURM_JOB_ID  GPUs: $SLURM_GPUS  CPUs: $SLURM_CPUS_ON_NODE  Mem: $SLURM_MEM_PER_NODE"')

print('\n=== venv active? ===')
venv_ok = str(VENV) in sys.executable
print(f'  {"✅ YES" if venv_ok else "❌ NO — run: source " + str(VENV) + "/bin/activate"}')

print('\n=== Lustre disk ===')
shell(f'df -h {LUSTRE_BASE} | tail -1')

Repo root : /lustre/smuexa01/client/users/mkotha/CS7325/Multi_LLM_as_Judge_Medical_AI
venv      : /lustre/smuexa01/client/users/mkotha/CS7325/.venv
HF cache  : /lustre/smuexa01/client/users/mkotha/CS7325/hf_models
Python    : /lustre/smuexa01/client/users/mkotha/CS7325/.venv/bin/python3
Started   : 2026-05-17T21:59:43.891680

=== GPU Status (expect 2x A100 80GB) ===
0, NVIDIA A100-SXM4-80GB, 81920 MiB, 81154 MiB
1, NVIDIA A100-SXM4-80GB, 81920 MiB, 81154 MiB
2, NVIDIA A100-SXM4-80GB, 81920 MiB, 81154 MiB
3, NVIDIA A100-SXM4-80GB, 81920 MiB, 81154 MiB

=== SLURM allocation ===
Node: bcm-dgxa100-0014  Job: 433186  GPUs: 4  CPUs: 64  Mem: 196608

=== venv active? ===
  ✅ YES

=== Lustre disk ===
100.127.24.16@o2ib1,100.127.25.16@o2ib1:100.127.24.17@o2ib1,100.127.25.17@o2ib1:100.127.24.18@o2ib1,100.127.25.18@o2ib1:100.127.24.19@o2ib1,100.127.25.19@o2ib1:/smuexa01  748T   37T  703T   5% /lustre/smuexa01/client


0

---
## Phase 2 — Install Dependencies

In [2]:
# CELL 2: Install all deps + auto-fix missing packages
REQUIREMENTS = ROOT / 'requirements.txt'
print(f'Installing from: {REQUIREMENTS}')
shell(f'{sys.executable} -m pip install -r {REQUIREMENTS} -q')

print('\nInstalling / verifying vLLM...')
shell(f'{sys.executable} -m pip install vllm -q')

# Auto-fix known missing packages
ALWAYS_INSTALL = ['tabulate', 'ipywidgets']
for pkg in ALWAYS_INSTALL:
    shell(f'{sys.executable} -m pip install {pkg} -q')

# Verify
REQUIRED = ['vllm', 'httpx', 'datasets', 'huggingface_hub', 'pandas',
            'matplotlib', 'seaborn', 'dotenv', 'pytest', 'tabulate']
print('\n=== Package check ===')
all_ok = True
for pkg in REQUIRED:
    try:
        m = importlib.import_module(pkg.replace('-', '_'))
        ver = getattr(m, '__version__', '?')
        print(f'  ✅ {pkg:<20} {ver}')
    except ImportError:
        print(f'  ❌ {pkg} — FAILED')
        all_ok = False

print('\n✅ All packages OK' if all_ok else '\n❌ Some packages missing — re-run this cell')

Installing from: /lustre/smuexa01/client/users/mkotha/CS7325/Multi_LLM_as_Judge_Medical_AI/requirements.txt

Installing / verifying vLLM...

=== Package check ===
INFO 05-17 21:59:56 [__init__.py:239] Automatically detected platform cuda.
  ✅ vllm                 0.8.5
  ✅ httpx                0.28.1
  ✅ datasets             4.8.5
  ✅ huggingface_hub      0.30.2
  ✅ pandas               2.3.3
  ✅ matplotlib           3.10.9
  ✅ seaborn              0.13.2
  ✅ dotenv               ?
  ✅ pytest               9.0.3
  ✅ tabulate             0.10.0

✅ All packages OK


--- 
## Phase 3 — Download Judge Models

Models cached at `/lustre/.../hf_models/` — never fills `/home`.

| Judge | Model | Max context | GPU | Notes |
|---|---|---|---|---|
| medgemma  | `google/medgemma-4b-it`    | 4096 | GPU 0 | Gated — HF token required |
| biomistral| `BioMistral/BioMistral-7B` | 4096 | GPU 1 | |
| meditron  | `epfl-llm/meditron-7b`     | 2048 | GPU 0 | |
| medalpaca | `medalpaca/medalpaca-7b`   | 2048 | GPU 1 | Replaces BioMedLM (vLLM 0.8.5 incompatible) |


In [3]:
# CELL 3: Load .env + set HF cache
from dotenv import load_dotenv
load_dotenv(ROOT / '.env')

os.environ['HF_HOME']            = str(HF_CACHE)
os.environ['TRANSFORMERS_CACHE'] = str(HF_CACHE)
os.environ['HF_DATASETS_CACHE']  = str(HF_CACHE / 'datasets')

HF_TOKEN = os.environ.get('HF_TOKEN')
GH_TOKEN = os.environ.get('GITHUB_TOKEN')

print(f'HF_TOKEN:      {"✅ set" if HF_TOKEN else "❌ NOT SET — add to .env"}')
print(f'GITHUB_TOKEN:  {"✅ set" if GH_TOKEN else "⚠️  not set — needed for git push"}')
print(f'HF_CACHE:      {HF_CACHE}')
shell(f'du -sh {HF_CACHE} 2>/dev/null || echo "(empty)"')

HF_TOKEN:      ✅ set
GITHUB_TOKEN:  ✅ set
HF_CACHE:      /lustre/smuexa01/client/users/mkotha/CS7325/hf_models
179G	/lustre/smuexa01/client/users/mkotha/CS7325/hf_models


0

In [4]:
# CELL 4: Download all judge models (skips already-cached)
# NOTE: BioMedLM (stanford-crfm/BioMedLM) is NOT downloaded —
#       it has a hard incompatibility with vLLM 0.8.5
#       (AssertionError: scale_attn_by_inverse_layer_idx not supported).
#       medalpaca/medalpaca-7b is used instead as Judge 4.
from huggingface_hub import snapshot_download
import time

JUDGE_MODELS = [
    {'id': 'medgemma',   'hf_id': 'google/medgemma-4b-it',       'needs_token': True},
    {'id': 'biomistral', 'hf_id': 'BioMistral/BioMistral-7B',    'needs_token': False},
    {'id': 'meditron',   'hf_id': 'epfl-llm/meditron-7b',        'needs_token': False},
    {'id': 'medalpaca',  'hf_id': 'medalpaca/medalpaca-7b',       'needs_token': False},
    # BioMedLM removed — incompatible with vLLM 0.8.5
    # {'id': 'biomedlm', 'hf_id': 'stanford-crfm/BioMedLM', ...}
]

for m in JUDGE_MODELS:
    marker_dir = HF_CACHE / f'models--{m["hf_id"].replace("/", "--")}'
    if marker_dir.exists() and any(marker_dir.rglob('config.json')):
        print(f'✅ [{m["id"]:12s}] already cached → {marker_dir}')
        continue
    if m['needs_token'] and not HF_TOKEN:
        print(f'❌ [{m["id"]:12s}] requires HF_TOKEN — set in .env')
        continue
    print(f'\n⏬  Downloading [{m["id"]}] {m["hf_id"]} ...')
    t0 = time.time()
    try:
        snapshot_download(
            repo_id=m['hf_id'],
            cache_dir=str(HF_CACHE),
            token=HF_TOKEN if m['needs_token'] else None,
            ignore_patterns=['*.msgpack', '*.h5', 'flax_model*', 'tf_model*', 'rust_model*'],
        )
        print(f'  ✅ Done in {(time.time()-t0)/60:.1f} min')
    except Exception as e:
        print(f'  ❌ Failed: {e}')

print('\n=== Total model cache ===')
shell(f'du -sh {HF_CACHE}')


✅ [medgemma    ] already cached → /lustre/smuexa01/client/users/mkotha/CS7325/hf_models/models--google--medgemma-4b-it
✅ [biomistral  ] already cached → /lustre/smuexa01/client/users/mkotha/CS7325/hf_models/models--BioMistral--BioMistral-7B
✅ [meditron    ] already cached → /lustre/smuexa01/client/users/mkotha/CS7325/hf_models/models--epfl-llm--meditron-7b
✅ [medalpaca   ] already cached → /lustre/smuexa01/client/users/mkotha/CS7325/hf_models/models--medalpaca--medalpaca-7b

=== Total model cache ===
179G	/lustre/smuexa01/client/users/mkotha/CS7325/hf_models


0

---
## Phase 4 — Launch vLLM Servers

**4x A100 80GB available — 1 dedicated GPU per model. No memory sharing.**

| Judge | Model | GPU | Port | Max Len | GPU Mem |
|---|---|---|---|---|---|
| medgemma  | `google/medgemma-4b-it`    | GPU 0 | 8001 | 4096 | 85% (~68GB) |
| biomistral| `BioMistral/BioMistral-7B` | GPU 1 | 8002 | 4096 | 85% (~68GB) |
| meditron  | `epfl-llm/meditron-7b`     | GPU 2 | 8003 | 2048 | 85% (~68GB) |
| medalpaca | `medalpaca/medalpaca-7b`   | GPU 3 | 8004 | 2048 | 85% (~68GB) |

All models use `HF_HUB_OFFLINE=1` + local snapshot paths — no HF auth required at runtime.
BioMedLM replaced with medalpaca (BioMedLM has hard vLLM 0.8.5 incompatibility).


In [ ]:
# CELL 5: Kill any old vLLM processes, then launch 4 fresh servers sequentially
import time, subprocess, os
from pathlib import Path

LUSTRE_BASE = Path('/lustre/smuexa01/client/users/mkotha/CS7325')
HF_CACHE    = LUSTRE_BASE / 'hf_models'
LOG_DIR     = LUSTRE_BASE / 'vllm_logs'
PYTHON      = str(LUSTRE_BASE / '.venv' / 'bin' / 'python')

LOG_DIR.mkdir(parents=True, exist_ok=True)

# ── Step 1: Full cleanup ─────────────────────────────────────────────────────
print('=' * 60)
print('Step 1: Killing all existing vLLM processes...')
subprocess.run('pkill -9 -f "vllm.entrypoints" 2>/dev/null || true', shell=True)
time.sleep(3)
subprocess.run(
    'kill -9 $(fuser /dev/nvidia0 /dev/nvidia1 /dev/nvidia2 /dev/nvidia3 2>/dev/null) 2>/dev/null || true',
    shell=True
)
time.sleep(8)
print('GPU memory after cleanup:')
subprocess.run(
    'nvidia-smi --query-gpu=index,memory.used,memory.free --format=csv,noheader',
    shell=True
)

# ── Step 2: Resolve local snapshot paths ────────────────────────────────────
print('\nStep 2: Resolving local model snapshot paths...')

def get_snapshot_path(hf_id):
    """Find the local cached snapshot directory for a HuggingFace model."""
    cache_name = 'models--' + hf_id.replace('/', '--')
    snap_dir = HF_CACHE / cache_name / 'snapshots'
    if not snap_dir.exists():
        raise FileNotFoundError(
            f"Model not cached: {hf_id}\n"
            f"Expected at: {snap_dir}\n"
            f"Run Cell 4 to download first."
        )
    snapshots = list(snap_dir.iterdir())
    if not snapshots:
        raise FileNotFoundError(f"Snapshot dir empty for {hf_id}: {snap_dir}")
    return str(sorted(snapshots, key=lambda p: p.stat().st_mtime)[-1])

# ── Judge configs — 4 GPUs, 1 model each — no memory sharing needed ─────────
# You have 4x A100 80GB. Each model gets its own dedicated GPU at 0.85
# utilization = ~68GB KV cache budget. No splitting, no OOM.
JUDGE_CONFIGS = [
    {
        'id':      'medgemma',
        'hf_id':   'google/medgemma-4b-it',
        'port':    8001,
        'gpu':     '0',           # dedicated GPU 0
        'max_len': 4096,
        'gpu_mem': 0.85,
        'extra':   '--trust-remote-code',
    },
    {
        'id':      'biomistral',
        'hf_id':   'BioMistral/BioMistral-7B',
        'port':    8002,
        'gpu':     '1',           # dedicated GPU 1
        'max_len': 4096,
        'gpu_mem': 0.85,
        'extra':   '',
    },
    {
        'id':      'meditron',
        'hf_id':   'epfl-llm/meditron-7b',
        'port':    8003,
        'gpu':     '2',           # dedicated GPU 2
        'max_len': 2048,
        'gpu_mem': 0.85,
        'extra':   '',
    },
    {
        'id':      'medalpaca',
        'hf_id':   'medalpaca/medalpaca-7b',
        'port':    8004,
        'gpu':     '3',           # dedicated GPU 3
        'max_len': 2048,
        'gpu_mem': 0.85,
        'extra':   '',
    },
]

# Resolve paths — fail fast if any model missing from cache
snapshot_paths = {}
all_cached = True
for j in JUDGE_CONFIGS:
    try:
        snap = get_snapshot_path(j['hf_id'])
        snapshot_paths[j['id']] = snap
        print(f"  ✅ [{j['id']:12s}] {snap.split('/')[-1][:20]}...")
    except FileNotFoundError as e:
        print(f"  ❌ [{j['id']:12s}] {e}")
        all_cached = False

if not all_cached:
    raise RuntimeError("Some models not cached — run Cell 4 first.")

# ── Step 3: Sequential launch with health-check gates ───────────────────────
print('\nStep 3: Launching judges (1 per GPU, sequential with health checks)...')
print('~3-5 min per model. Total: ~15-20 min.\n')

def wait_healthy(port, name, timeout=600):
    import urllib.request
    url = f'http://localhost:{port}/health'
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if r.status == 200:
                    return True
        except Exception:
            pass
        time.sleep(10)
        elapsed = time.time() - t0
        print(f'    ⏳ [{name}] waiting... {elapsed:.0f}s', end='\r')
    return False

def launch_judge(j, snap_path):
    log_file = str(LOG_DIR / f'vllm_{j["id"]}.log')
    subprocess.run(f'fuser -k {j["port"]}/tcp 2>/dev/null || true', shell=True)
    time.sleep(2)

    cmd = (
        f'CUDA_VISIBLE_DEVICES={j["gpu"]} '
        f'HF_HUB_OFFLINE=1 '
        f'TRANSFORMERS_OFFLINE=1 '
        f'HF_HOME={HF_CACHE} '
        f'{PYTHON} -m vllm.entrypoints.openai.api_server '
        f'--model {snap_path} '
        f'--port {j["port"]} '
        f'--host 0.0.0.0 '
        f'--gpu-memory-utilization {j["gpu_mem"]} '
        f'--max-model-len {j["max_len"]} '
        f'--dtype bfloat16 '
        f'{j["extra"]} '
        f'> {log_file} 2>&1'
    )
    proc = subprocess.Popen(cmd, shell=True)
    print(f'  🚀 [{j["id"]:12s}] PID={proc.pid} GPU={j["gpu"]} port={j["port"]} mem={j["gpu_mem"]}')

    ok = wait_healthy(j['port'], j['id'], timeout=600)

    if ok:
        print(f'  ✅ [{j["id"]:12s}] UP on port {j["port"]}                    ')
        subprocess.run(
            'nvidia-smi --query-gpu=index,memory.used,memory.free --format=csv,noheader',
            shell=True
        )
    else:
        print(f'  ❌ [{j["id"]:12s}] FAILED — last 15 log lines:')
        subprocess.run(f'tail -15 {log_file}', shell=True)
        raise RuntimeError(
            f"Judge {j['id']} failed to start. See log above.\n"
            f"Full log: {log_file}"
        )
    return proc

procs = {}
for j in JUDGE_CONFIGS:
    print(f'\n--- Launching {j["id"]} (GPU {j["gpu"]}, port {j["port"]}) ---')
    procs[j['id']] = launch_judge(j, snapshot_paths[j['id']])

print('\n' + '=' * 60)
print('✅ ALL 4 JUDGES UP — proceed to Cell 7 to confirm')
print('=' * 60)

JUDGE_URLS = {j['id']: f'http://localhost:{j["port"]}' for j in JUDGE_CONFIGS}


Step 1: Killing all existing vLLM processes...
GPU memory after cleanup:
0, 0 MiB, 81154 MiB
1, 0 MiB, 81154 MiB
2, 0 MiB, 81154 MiB
3, 0 MiB, 81154 MiB

Step 2: Resolving local model snapshot paths...
  ✅ [medgemma    ] 290cda5eeccbee130f98...
  ✅ [biomistral  ] 9a11e1ffa817c211cbb5...
  ✅ [meditron    ] d7d0a5ed929384a6b059...
  ✅ [medalpaca   ] fbb41b75d5a46ba405d4...

Step 3: Launching judges (1 per GPU, sequential with health checks)...
~3-5 min per model. Total: ~15-20 min.


--- Launching medgemma (GPU 0, port 8001) ---
  🚀 [medgemma    ] PID=2342191 GPU=0 port=8001 mem=0.85
    ⏳ [medgemma] waiting... 70s

In [ ]:
# CELL 6 (optional): Check last 8 lines of each vLLM log
# Run anytime to see startup progress or diagnose errors
if 'JUDGE_CONFIGS' not in dir():
    # Fallback if run independently
    JUDGE_CONFIGS = [
        {'id': 'medgemma',   'gpu': '0', 'port': 8001},
        {'id': 'biomistral', 'gpu': '1', 'port': 8002},
        {'id': 'meditron',   'gpu': '0', 'port': 8003},
        {'id': 'medalpaca',  'gpu': '1', 'port': 8004},
    ]

for j in JUDGE_CONFIGS:
    log_file = LOG_DIR / f'vllm_{j["id"]}.log'
    print(f'\n--- [{j["id"]}] GPU={j["gpu"]} port={j["port"]} ---')
    shell(f'tail -8 {log_file} 2>/dev/null || echo "(no log yet)"')


---
## Phase 5 — Health Check (wait for all 4 judges)

In [ ]:
# CELL 7: Poll until all 4 vLLM servers respond — 20 min max
import httpx, time

# JUDGE_URLS set by Cell 5; fallback to defaults if re-running independently
if 'JUDGE_URLS' not in dir():
    JUDGE_URLS = {
        'medgemma':   os.getenv('LOCAL_VLLM_JUDGE1_URL', 'http://localhost:8001'),
        'biomistral': os.getenv('LOCAL_VLLM_JUDGE2_URL', 'http://localhost:8002'),
        'meditron':   os.getenv('LOCAL_VLLM_JUDGE3_URL', 'http://localhost:8003'),
        'medalpaca':  os.getenv('LOCAL_VLLM_JUDGE4_URL', 'http://localhost:8004'),
    }

MAX_WAIT = 1200  # 20 min
POLL     = 30
ready    = {k: False for k in JUDGE_URLS}
t_start  = time.time()

print('Checking all 4 vLLM judges...')
print(f'Max wait: {MAX_WAIT//60} min | Poll: {POLL}s\n')

while not all(ready.values()):
    elapsed = time.time() - t_start
    if elapsed > MAX_WAIT:
        print(f'\n⏱️  Timeout. Checking logs for failures:')
        if 'JUDGE_CONFIGS' in dir():
            for j in JUDGE_CONFIGS:
                if not ready[j['id']]:
                    print(f'\n--- [{j["id"]}] last 15 lines ---')
                    shell(f'tail -15 {LOG_DIR}/vllm_{j["id"]}.log')
        break
    for name, url in JUDGE_URLS.items():
        if ready[name]: continue
        try:
            r = httpx.get(f'{url}/health', timeout=4)
            if r.status_code == 200:
                ready[name] = True
                print(f'  ✅ {name:12s} {url}  [{elapsed:.0f}s]')
        except Exception:
            pass
    if not all(ready.values()):
        pending = [k for k, v in ready.items() if not v]
        print(f'  ⏳ Pending: {pending}  [{elapsed:.0f}s]')
        time.sleep(POLL)

if all(ready.values()):
    elapsed = time.time() - t_start
    print(f'\n✅ ALL 4 JUDGES READY in {elapsed/60:.1f} min — proceed to experiments')
    print('\nGPU memory allocation:')
    shell('nvidia-smi --query-gpu=index,memory.used,memory.free --format=csv,noheader')
else:
    failed = [k for k, v in ready.items() if not v]
    print(f'\n❌ Failed judges: {failed}')
    print('Fix errors above, then re-run Cell 5')


---
## Phase 6 — Pre-Flight Checks

In [ ]:
# CELL 8: Verify all required repo files exist
import json

REQUIRED_PATHS = [
    ROOT / 'core'        / 'wrapper.py',
    ROOT / 'core'        / 'agreement.py',
    ROOT / 'core'        / 'rubric_engine.py',
    ROOT / 'core'        / 'metrics.py',
    ROOT / 'core'        / 'schemas.py',
    ROOT / 'core'        / 'consensus_core' / 'models.py',
    ROOT / 'rubrics'     / 'rubric1_pemat.json',
    ROOT / 'rubrics'     / 'rubric2_healthbench.json',
    ROOT / 'rubrics'     / 'rubric3_clinical_eval.json',
    ROOT / 'rubrics'     / 'rubric4_prometheus.json',
    ROOT / 'config'      / 'configs' / 'config_exp1_dataset.json',
    ROOT / 'config'      / 'configs' / 'config_exp2_agreement.json',
    ROOT / 'experiments' / 'exp1_dataset_analysis.py',
    ROOT / 'experiments' / 'exp2_agreement_analysis.py',
    ROOT / 'experiments' / 'exp3_rubric_sensitivity.py',
    ROOT / 'experiments' / 'exp4_boxplot_agreement.py',
]

print('=== Repo structure check ===')
all_ok = True
for p in REQUIRED_PATHS:
    ok = p.exists()
    print(f'  {"✅" if ok else "❌"} {p.relative_to(ROOT)}')
    if not ok: all_ok = False

if not all_ok:
    print(f'\n❌ Missing files — run: git -C {ROOT} pull origin main')
else:
    print('\n✅ All files present')

In [ ]:
# CELL 9: Run unit tests
print('Running pytest...')
rc = shell(f'{sys.executable} -m pytest {ROOT}/tests/test_core.py -v --tb=short')
print('\n✅ All tests passed' if rc == 0 else '\n⚠️  Tests failed — review before running experiments')

---
## Phase 7 — Experiment 1: ADRD-Bench Dataset Analysis

In [ ]:
# CELL 10: Experiment 1
import importlib, experiments.exp1_dataset_analysis as exp1_mod
importlib.reload(exp1_mod)

print('Running Experiment 1: ADRD-Bench Dataset Analysis')
print('='*60)
exp1_mod.main()

from collections import Counter
import pandas as pd
q_path = ROOT / 'benchmark_dataset' / 'adrd_questions.json'
with open(q_path) as f:
    qdata = json.load(f)
counts = Counter(q['category'] for q in qdata['questions'])
df_cat = pd.DataFrame([{'Category': k, 'N': v} for k, v in sorted(counts.items())])
df_cat.loc[len(df_cat)] = ['TOTAL', df_cat['N'].sum()]
print('\n=== Dataset Breakdown ===')
print(df_cat.to_markdown(index=False))
print(f'\n✅ Exp1 complete — {qdata["metadata"]["total_questions"]} questions')

---
## Phase 8 — Experiment 2: Per-Rubric Agreement Analysis

In [ ]:
# CELL 11: Experiment 2
import experiments.exp2_agreement_analysis as exp2_mod
importlib.reload(exp2_mod)

print('Running Experiment 2: Per-Rubric Agreement Analysis')
print('='*60)
exp2_mod.main()

exp2_path = ROOT / 'results' / 'exp2_agreement_results.json'
with open(exp2_path) as f:
    exp2_data = json.load(f)

rows = []
for block in exp2_data:
    results = block['results']
    classes = Counter(r['agreement_class'] for r in results)
    total   = len(results)
    mean_pw = sum(r['mean_pairwise_agreement'] for r in results) / total
    rows.append({
        'Rubric':           block['rubric_name'].split('—')[0].strip()[:30],
        'N':                total,
        'Fully Agree %':    round(classes.get('fully_agree',    0)/total*100, 1),
        'Majority Agree %': round(classes.get('majority_agree', 0)/total*100, 1),
        'Split %':          round(classes.get('split',          0)/total*100, 1),
        'Disagree %':       round(classes.get('full_disagree',  0)/total*100, 1),
        'Mean PW%':         round(mean_pw, 1),
    })
df_exp2 = pd.DataFrame(rows)
print('\n=== Agreement Table ===')
print(df_exp2.to_markdown(index=False))
print(f'\n✅ Exp2 complete')

---
## Phase 9 — Experiment 3: Rubric Sensitivity

In [ ]:
# CELL 12: Experiment 3
import experiments.exp3_rubric_sensitivity as exp3_mod
importlib.reload(exp3_mod)

print('Running Experiment 3: Rubric Sensitivity')
print('='*60)
exp3_mod.main()

exp3_path = ROOT / 'results' / 'exp3_sensitivity_results.json'
with open(exp3_path) as f:
    exp3_data = json.load(f)

sens_rows = []
for block in exp3_data:
    for v in block['variants']:
        sens_rows.append({
            'Rubric':  block['rubric_name'].split('—')[0].strip()[:25],
            'Variant': v['scoring_variant'],
            'Mean%':   v['mean_pairwise_agreement'],
            'Std%':    v['std_pairwise_agreement'],
        })
print(pd.DataFrame(sens_rows).to_markdown(index=False))
print(f'\n✅ Exp3 complete')

---
## Phase 10 — Experiment 4: Box Plots

In [ ]:
# CELL 13: Experiment 4
import experiments.exp4_boxplot_agreement as exp4_mod
importlib.reload(exp4_mod)

print('Running Experiment 4: Box Plot Generation')
print('='*60)
exp4_mod.main()

pngs = sorted((ROOT / 'results' / 'figures').glob('*.png'))
print(f'\n✅ Exp4 complete: {len(pngs)} figures')
for p in pngs:
    print(f'  📊 {p.name}  ({p.stat().st_size//1024} KB)')

---
## Phase 11 — Results Summary

In [ ]:
# CELL 14: Print full results + save CSV
print('=== FINAL RESULTS — COPY TO PAPER ===\n')
print('[Exp 1] Dataset:')
for cat, n in sorted(counts.items()): print(f'  {cat}: {n}')
print(f'  TOTAL: {sum(counts.values())}')
print('\n[Exp 2] Agreement Table:')
print(df_exp2.to_markdown(index=False))
print('\n[Exp 3] Best scoring variant per rubric:')
for block in exp3_data:
    best = max(block['variants'], key=lambda v: v['mean_pairwise_agreement'])
    print(f'  {block["rubric_name"][:30]:30s} → {best["scoring_variant"]} ({best["mean_pairwise_agreement"]:.1f}%)')
print(f'\n[Exp 4] {len(pngs)} PNG figures in results/figures/')
df_exp2.to_csv(ROOT / 'results' / 'agreement_summary_table.csv', index=False)
print('\n✅ agreement_summary_table.csv saved')

---
## Phase 12 — Display Figures Inline

In [ ]:
# CELL 15: Display all PNG figures
from IPython.display import Image, display, Markdown
for png in sorted((ROOT / 'results' / 'figures').glob('*.png')):
    display(Markdown(f'### {png.stem}'))
    display(Image(filename=str(png), width=950))

---
## Phase 13 — Push Results to GitHub

In [ ]:
# CELL 16: Commit and push results
if GH_TOKEN:
    shell(f'git -C {ROOT} remote set-url origin https://{GH_TOKEN}@github.com/m22oct2000/Multi_LLM_as_Judge_Medical_AI.git')
else:
    print('⚠️  GITHUB_TOKEN not set — push will prompt for password')

COMMIT_MSG = f'HPC results (SuperPOD mkotha): all 4 experiments — {datetime.now().strftime("%Y-%m-%d %H:%M")}'

print('=== Files to commit ===')
shell(f'git -C {ROOT} status --short')

shell(f'git -C {ROOT} add results/')
shell(f'git -C {ROOT} add benchmark_dataset/')
shell(f'git -C {ROOT} commit -m "{COMMIT_MSG}"')

rc = shell(f'git -C {ROOT} push origin main')
if rc == 0:
    print('\n✅ Pushed! → https://github.com/m22oct2000/Multi_LLM_as_Judge_Medical_AI/tree/main/results')
else:
    print(f'\n⚠️  Push failed. Try: cd {ROOT} && git push origin main')

---
## Phase 14 — Cleanup

In [ ]:
# CELL 17: Kill vLLM servers and free all GPUs
print('=== GPU before cleanup ===')
shell('nvidia-smi --query-gpu=index,memory.used,memory.free --format=csv,noheader')

if 'procs' in dir():
    for name, proc in procs.items():
        try:
            proc.terminate()
            print(f'  Terminated [{name}] PID={proc.pid}')
        except Exception:
            pass

subprocess.run('pkill -9 -f "vllm.entrypoints" 2>/dev/null || true', shell=True)
time.sleep(3)
subprocess.run(
    'kill -9 $(fuser /dev/nvidia0 /dev/nvidia1 /dev/nvidia2 /dev/nvidia3 2>/dev/null) 2>/dev/null || true',
    shell=True
)
import time; time.sleep(4)
print('\n=== GPU after cleanup ===')
shell('nvidia-smi --query-gpu=index,memory.used,memory.free --format=csv,noheader')
print('\n✅ Done. All 4 GPUs free. Exit srun with: exit')


---
## ✅ All Experiments Complete

| File | Use |
|---|---|
| `results/agreement_summary_table.csv` | Paper Table 2 |
| `results/figures/*.png` | Upload to Overleaf |
| `results/exp2_agreement_results.json` | Full rationales |
| `benchmark_dataset/adrd_questions.json` | Dataset reference |

**Acknowledgement for paper:**
> *"Computational resources were provided by SMU's O'Donnell Data Science and Research Computing Institute (SuperPOD cluster, 2x NVIDIA A100 80GB GPUs, allocation: xnluo_ai_chatbot_cognitive_0002)."*
